# MedEHA t-SNE 特征可视化

参考 CoMA `visualize_t-sne.ipynb`：在 CheXpert 验证集上提取图像全局特征，对比 **Baseline** 与 **MedEHA** 预训练模型的 t-SNE 分布。

**使用前请修改** 两个 checkpoint 路径与采样参数。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

from medeha.constants import CHEXPERT_COMPETITION_TASKS
from medeha.datasets.transforms import Moco2Transform
from medeha.datasets.data_module import DataModule
from medeha.datasets.classification_dataset import CheXpertImageDataset
from medeha.pretrain.module import MedEHA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 72
IMG_SIZE = 224

In [ ]:
# ===== 用户配置 =====
BASELINE_CKPT = "path/to/baseline/checkpoint.ckpt"
MEDEHA_CKPT = "path/to/medeha/checkpoint.ckpt"
DATA_PCT = 1.0
MAX_SAMPLES_PER_CLASS = 1000
TSNE_PERPLEXITY = 20

datamodule = DataModule(
    CheXpertImageDataset,
    None,
    Moco2Transform,
    data_pct=DATA_PCT,
    batch_size=BATCH_SIZE,
    num_workers=8,
)
dataloader = datamodule.val_dataloader()

baseline_model = MedEHA.load_from_checkpoint(BASELINE_CKPT, strict=False).to(device).eval()
medeha_model = MedEHA.load_from_checkpoint(MEDEHA_CKPT, strict=False).to(device).eval()

In [ ]:
def extract_features(model, dataloader, device, max_batches=None):
    features_list, labels_list = [], []
    for i, batch in enumerate(tqdm(dataloader, desc="extract")):
        x, y = batch
        x = x.to(device)
        with torch.no_grad():
            img_feat, _, _ = model.img_encoder_q(x, None)
            if img_feat.dim() > 2:
                img_feat = img_feat.view(img_feat.size(0), -1)
        features_list.append(img_feat.cpu().numpy())
        labels_list.append(np.array(y))
        if max_batches is not None and i + 1 >= max_batches:
            break
    return np.vstack(features_list), np.vstack(labels_list)


def sample_single_label(features_a, features_b, labels, n_sample=None):
    mask = np.array([np.sum(s) == 1 for s in labels])
    features_a = features_a[mask]
    features_b = features_b[mask]
    labels = labels[mask]
    class_ids = np.argmax(labels, axis=1)

    if n_sample is not None:
        keep = []
        counts = np.zeros(labels.shape[1], dtype=int)
        for cid in class_ids:
            if counts[cid] < n_sample:
                keep.append(True)
                counts[cid] += 1
            else:
                keep.append(False)
        keep = np.array(keep)
        features_a = features_a[keep]
        features_b = features_b[keep]
        class_ids = class_ids[keep]
    return features_a, features_b, class_ids


def run_tsne(features_a, features_b, perplexity=30):
    tsne_a = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    tsne_b = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    return tsne_a.fit_transform(features_a), tsne_b.fit_transform(features_b)


def plot_tsne_comparison(f2d_baseline, f2d_medeha, labels, save_path=None):
    unique_labels = sorted(np.unique(labels))
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
    color_map = {label: colors[i] for i, label in enumerate(unique_labels)}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    for label in unique_labels:
        mask = np.array(labels) == label
        name = CHEXPERT_COMPETITION_TASKS[label]
        ax1.scatter(f2d_baseline[mask, 0], f2d_baseline[mask, 1], c=[color_map[label]], label=name, alpha=0.7, s=30)
        ax2.scatter(f2d_medeha[mask, 0], f2d_medeha[mask, 1], c=[color_map[label]], label=name, alpha=0.7, s=30)

    ax1.set_title("Baseline", fontsize=24)
    ax2.set_title("MedEHA", fontsize=24)
    for ax in (ax1, ax2):
        ax.set_xlabel("t-SNE Component 1", fontsize=16)
        ax.set_ylabel("t-SNE Component 2", fontsize=16)
        ax.legend(loc="upper right", fontsize=11)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
baseline_features, baseline_labels = extract_features(baseline_model, dataloader, device)
medeha_features, medeha_labels = extract_features(medeha_model, dataloader, device)

if not np.array_equal(baseline_labels, medeha_labels):
    raise ValueError("两个模型提取的标签不一致，请检查 dataloader 顺序。")

n_baseline, n_medeha, n_labels = sample_single_label(
    baseline_features, medeha_features, baseline_labels, n_sample=MAX_SAMPLES_PER_CLASS
)
print(f"采样后样本数: {len(n_labels)}")

In [ ]:
f2d_baseline, f2d_medeha = run_tsne(n_baseline, n_medeha, perplexity=TSNE_PERPLEXITY)
plot_tsne_comparison(f2d_baseline, f2d_medeha, n_labels, save_path="tsne_medeha_vs_baseline.png")